# Concurrent MiniLM-L6 masking and collapse-guardrail test

This notebook documents the current live training experiment and reads its local outputs after the Colab transfer completes. Two full-data fine-tuning workers run concurrently in one T4 session from the Git-shipped `all-MiniLM-L6-v2` base model; neither worker resumes a checkpoint.

The experiment compares the masking change independently from the collapse-guardrail operating threshold:

- Worker 1: `matched_positive_negative` masking, collapse threshold `0.80`.
- Worker 2: `baseline` masking, collapse threshold `0.65`.

The cells below intentionally report pending status when the run has not transferred yet; they do not manufacture metrics.

In [ ]:
from pathlib import Path
import json
import re
import yaml
import pandas as pd

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'config' / 'training.yaml').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'config' / 'training.yaml').is_file():
    raise FileNotFoundError('could not locate project config/training.yaml')
RESULTS = ROOT / 'training_results'
CONFIG = yaml.safe_load((ROOT / 'config' / 'training.yaml').read_text())
print({'project': str(ROOT), 'results_root': str(RESULTS)})

## Configuration captured for this test

This is read from the SSOT configuration rather than duplicated in the training code.

In [ ]:
import sys
sys.path.insert(0, str(ROOT / 'src'))
from core.common import masking_cfg, collapse_guardrail_cfg
modified_masking = masking_cfg('matched_positive_negative')
baseline_masking = masking_cfg('baseline')
threshold_80 = collapse_guardrail_cfg('threshold_80')
threshold_65 = collapse_guardrail_cfg('threshold_65')
experiment = pd.DataFrame([
    {
        'worker': 1,
        'model': 'minilm_l6',
        'model_source': 'Git-shipped artifacts/models/all-MiniLM-L6-v2',
        'masking_profile': 'matched_positive_negative',
        'mask_lo': modified_masking['mask_lo'],
        'mask_hi': modified_masking['mask_hi'],
        'hard_negative_frac': modified_masking['hard_negative_frac'],
        'guardrail_profile': 'threshold_80',
        'operating_threshold': threshold_80['operating_threshold'],
        'data': 'full',
        'starting_checkpoint': 'none',
    },
    {
        'worker': 2,
        'model': 'minilm_l6',
        'model_source': 'Git-shipped artifacts/models/all-MiniLM-L6-v2',
        'masking_profile': 'baseline',
        'mask_lo': baseline_masking['mask_lo'],
        'mask_hi': baseline_masking['mask_hi'],
        'hard_negative_frac': baseline_masking['hard_negative_frac'],
        'guardrail_profile': 'threshold_65',
        'operating_threshold': threshold_65['operating_threshold'],
        'data': 'full',
        'starting_checkpoint': 'none',
    },
])
experiment

In [ ]:
runs = sorted(RESULTS.glob('concurrent_train_*'), key=lambda p: p.stat().st_mtime)
if not runs:
    raise FileNotFoundError(f'no concurrent training output under {RESULTS}')
run_dir = runs[-1]
workers = sorted(run_dir.glob('worker_*'), key=lambda p: int(re.search(r'\d+', p.name).group()))
print({'run_dir': str(run_dir), 'workers_found': [p.name for p in workers]})

def read_status(worker):
    status = worker / 'training.status'
    return status.read_text().strip() if status.is_file() else 'running_or_not_transferred'

pd.DataFrame([{'worker': p.name, 'status': read_status(p), 'log': (p / 'training.log').is_file()} for p in workers])

## Worker evidence

The next cell scans the durable logs and JSON outputs for collapse, threshold, Rand, ARI, precision, recall, over-merge, under-merge, and unmatched-SKU diagnostics. Missing values mean that the run has not emitted or transferred that artifact yet; they are not converted to zero.

In [ ]:
KEYS = ('collapse', 'threshold', 'rand', 'ari', 'precision', 'recall', 'over_merge', 'under_merge', 'unmatched')
def evidence(worker):
    text = '\n'.join(p.read_text(errors='replace') for p in worker.glob('*.log') if p.is_file())
    json_objects = []
    for path in worker.rglob('*.json'):
        try:
            value = json.loads(path.read_text())
        except (OSError, json.JSONDecodeError):
            continue
        if isinstance(value, dict):
            json_objects.append((path.name, value))
    flat = {}
    def visit(value, prefix=''):
        if isinstance(value, dict):
            for key, child in value.items():
                visit(child, f'{prefix}.{key}' if prefix else key)
        elif any(key in prefix.lower() for key in KEYS):
            flat[prefix] = value
    for name, value in json_objects:
        visit(value, name)
    log_lines = [line for line in text.splitlines() if any(key in line.lower() for key in KEYS)]
    return {'worker': worker.name, 'status': read_status(worker), 'json_metrics': flat, 'log_evidence': log_lines[-40:]}

evidence_rows = [evidence(worker) for worker in workers]
for row in evidence_rows:
    print(f"\n=== {row['worker']} status={row['status']} ===")
    print(json.dumps(row['json_metrics'], indent=2, default=str))
    print('\n'.join(row['log_evidence']))

## Conclusions

The valid conclusion is deliberately conditional on both workers finishing and emitting the guardrail and calibration metrics. Compare the workers only after confirming: (1) both used the expected profile, (2) both started from the shipped base, (3) collapse guardrail status is `passed`/equivalent rather than skipped, and (4) Rand/ARI and the GTIN-stratified metrics are present.

A higher Rand Index alone is not sufficient if the corresponding worker has a collapse violation, missing GTIN strata, or a failed/partial result transfer.

In [ ]:
complete = all(row['status'] == '0' for row in evidence_rows) and len(evidence_rows) == 2
print({
    'run_complete': complete,
    'interpretation': (
        'Both workers completed; inspect the evidence above and compare only metrics produced by both.'
        if complete else
        'Run is pending or incomplete; no performance conclusion is valid yet.'
    ),
})